In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.chdir('drive/MyDrive/데구론')

In [ ]:
import numpy as np
from cvxopt import matrix, solvers
import cvxopt
import codecs
from time import time


f_point = codecs.open('point_2x0.txt', 'r',encoding='utf8')
f_return = codecs.open('matrix_cp_return.txt', 'r', encoding='utf8')
f_vars = open('vars.txt', 'r') #vars와 같은 파일.

point_2x0 = []
cp_return_s = []
vars = []
cp_mean =np.zeros((3000, 1))

In [ ]:
#point_2x0.txt 읽고 배열로 만드는 과정
line_point = f_point.readline()
for i in range(0, 3000):
    line_point = f_point.readline()
    line_point = line_point.replace("\n", "");
    pushData_point = line_point.split('\t')
    point_2x0.append(float(pushData_point[1]))


In [ ]:
#matrix_cp_return.txt 읽고 배열로 만든 후 행렬에 넣는 과정
line_return= f_return.readline() # 텍스트 파일에 첫 번째 줄은 x1과 같은 값들이기 때문에 필요 없음.
line_return= f_return.readline()
line_return = line_return.replace("\n", "");
pushData_return = line_return.split('\t')
for i in range(0,3000):
    cp_return_s.append(float(pushData_return[i]))
cp_return = matrix(cp_return_s) #lp를 solve하기 위해 matrix로 바꿔줌.


In [ ]:
# 파일 열기
with codecs.open('matrix_cp_mean.txt', 'r', encoding='utf8') as file:
    # numpy 배열로 데이터 로드
    data = np.loadtxt(file, delimiter="\t",skiprows=1)

# 컬럼 별 합계 계산
column_sums = np.sum(data, axis=0)

# 결과를 Ldata에 할당
data = column_sums

np.savetxt('Ldata.txt', data, delimiter='\t')
f_Ldata = codecs.open('Ldata.txt', 'r', encoding='utf8') #cp_mean을 칼럼썸한 파일.

with open('Ldata.txt', 'r') as file:
    # 파일에서 모든 행을 읽고, 각 행의 값을 float으로 변환하여 cp_mean 배열에 할당
    cp_mean = np.array([float(line.strip()) for line in file])

# cp_mean 배열의 값들을 matrix 객체로 변환하고, 부호를 반전시킴
Cost = -matrix(cp_mean)

# 결과 출력 (옵션)
print(Cost)

In [ ]:
for i in range(0, 80000):
    line_vars = f_vars.readline()
    line_vars = line_vars.replace("\n",'')
    pushData_vars = line_vars.split('\t')
    temp=[] #vars라는 배열에 한번에 넣기 위해 편의상 생성한 임시 배열.
    for j in range(0,len(pushData_vars)-1):
            temp.append(int(pushData_vars[j]))
    vars.append(temp)

In [ ]:
len(vars[0])

1781

In [ ]:
#실제로 해결할 때는 메모리 초과로 인해 IF문을 사용해서 2만개씩 나누어 계산하였음.


f_point.close()
f_return.close()
f_Ldata.close()
f_vars.close()

# 데이터 값 정렬완료

In [ ]:
left_coefficient = np.concatenate((np.ones((1,3000)),-np.ones((1,3000)),-cp_return.T,-np.eye(3000, dtype=int),np.eye(3000, dtype=int)),axis=0)
#left_coefficient라는 배열은 좌변 계수를 의미. 이 매트릭스 안에는 1x3000행렬에 전부 1, 1x3000행렬에 전부 -1, cp.return의 트랜스포즈한 음수값, 3000*3000의 음수 단위행렬, 3000*3000 단위행렬이 들어감

left_coefficient=matrix(left_coefficient) #LP Solve를 위해 행렬로 변환.


#create full RHS
constant=-0.0963007951203275
right_constraint = matrix([1,-1,constant,matrix(np.zeros((3000,1))),matrix(point_2x0)])
#위에서부터 1, -1, constant, 3000*1 전부 0, point_2x0 값이 들어가 있음.

In [ ]:

f_time = open('time_results.txt', mode='w', encoding='utf-8')
f_status = open('status_results.txt', mode='w', encoding='utf-8')
f_variable_count = open('variable_count_results.txt', mode='w', encoding='utf-8')
f_objective_value = open('objective_value_results.txt', mode='w', encoding='utf-8')
f_variable_values = open('variable_values_results.txt', mode='w', encoding='utf-8')


variable_names = ['x' + str(i + 1) for i in range(3000)]
f_variable_values.write(' '.join(variable_names) + '\n')
result = []
for i in range(0,100): # 8만개 해결
    from time import time #for문 돌릴때마다 초기화시켜줘야 함.
    start = time() #시작 시간
    cons=[0,1,2] #constraint의 번호를 의미함. 0번은 budget조건, 1번은 -budget조건, 2번은 constraint의 조건.
    s_result=[] #한번 돌릴 때 결과값 저장하는 배열.

    for j in vars[i]:
        cons.append(j+3)
        cons.append(j+3003)

    s_left_c = left_coefficient[cons,vars[i]]
    s_right_c = right_constraint[cons]
    s_cost = Cost[vars[i]]
    sol=solvers.lp(s_cost,s_left_c,s_right_c, solver = 'glpk') #glpk를 사용해서 lp푸는 시간 최적화. glpk 사용 x = 하나당 최소 5초, glpk 사용 o = 하나당 최대 0.2초
    end=time() #끝난 시간
    time=end-start #걸린 시간.



    f_time.write(str(time) + '\n')
    f_status.write(str(sol['status']) + '\n')

    # 사용된 변수 개수 처리
    if sol['x'] is not None:
        f_variable_count.write(str(len(sol['x'])) + '\n')
    else:
        f_variable_count.write('0\n')  # 변수가 없으면 0으로 기록

    # 목적값 기록
    if sol['primal objective'] is not None:
        f_objective_value.write(str(sol['primal objective']) + '\n')
    else:
        f_objective_value.write('None\n')  # 목적값이 없으면 'None'으로 기록


    if sol['x'] is not None:
        variable_values = ['Null'] * 3000  # 모든 변수를 'Null'로 초기화
        for index, value in zip(vars[i], sol['x']):
            variable_values[index] = value  # 사용된 변수에 대한 값을 업데이트
    else:
        variable_values = ['Null'] * 3000  # 우선 모든 변수를 'Null'로 초기화
        for index in vars[i]:
            variable_values[index] = '-'  # 문제 해결에 사용된 변수에 대해서만 '-'로 표시

    variable_values_str = ' '.join([str(value) for value in variable_values])
    f_variable_values.write(variable_values_str + '\n')





    print(i+1,'번째 LP Solved')

#f_result.close()

1 번째 LP Solved
2 번째 LP Solved
3 번째 LP Solved
4 번째 LP Solved
5 번째 LP Solved
6 번째 LP Solved
7 번째 LP Solved
8 번째 LP Solved
9 번째 LP Solved
10 번째 LP Solved
11 번째 LP Solved
12 번째 LP Solved
13 번째 LP Solved
14 번째 LP Solved
15 번째 LP Solved
16 번째 LP Solved
17 번째 LP Solved
18 번째 LP Solved
19 번째 LP Solved
20 번째 LP Solved
21 번째 LP Solved
22 번째 LP Solved
23 번째 LP Solved
24 번째 LP Solved
25 번째 LP Solved
26 번째 LP Solved
27 번째 LP Solved
28 번째 LP Solved
29 번째 LP Solved
30 번째 LP Solved
31 번째 LP Solved
32 번째 LP Solved
33 번째 LP Solved
34 번째 LP Solved
35 번째 LP Solved
36 번째 LP Solved
37 번째 LP Solved
38 번째 LP Solved
39 번째 LP Solved
40 번째 LP Solved
41 번째 LP Solved
42 번째 LP Solved
43 번째 LP Solved
44 번째 LP Solved
45 번째 LP Solved
46 번째 LP Solved
47 번째 LP Solved
48 번째 LP Solved
49 번째 LP Solved
50 번째 LP Solved
51 번째 LP Solved
52 번째 LP Solved
53 번째 LP Solved
54 번째 LP Solved
55 번째 LP Solved
56 번째 LP Solved
57 번째 LP Solved
58 번째 LP Solved
59 번째 LP Solved
60 번째 LP Solved
61 번째 LP Solved
62 번째 LP Solved
63 번째 LP Solved
6